# 01 · One short call, one stock hedge, one cash ledger

**Objective:** follow a short European call from sale through expiry, accounting for the premium, stock trades, interest, transaction costs and final settlement. Then inspect the same hedge policy over 5,000 simulated outcomes.

The baseline selects **JPM**, rebalances every **five trading sessions**, and charges **5 basis points** on the absolute dollar value of each stock trade, including entry and liquidation. The input cell controls these choices.

This notebook is complete and runs independently from a fresh kernel. Its model inputs load from the same validated configuration as notebook 00; hedge-policy controls remain explicit below. No earlier notebook needs to be executed in the current kernel.

Prices follow illustrative correlated GBM under Q, with zero dividends and constant financing. Quantities are share-equivalent option units; there is no exchange contract multiplier. Results describe this simulation, rather than a historical trading record.

**Shared study inputs:** contract and scenario assumptions now load from `config/illustrative_study.json` through `load_named_inputs`. `Book` remains the validated model container. Change that file once to update the baseline across notebooks 00–03; policy/experiment controls stay explicit in each notebook. 


| Step | Purpose | Main functions/tools |
|---|---|---|
| 1–2 | Load packages and inspect named inputs | `Path`, dictionaries, `Book` |
| 3 | Reproduce the joint study paths and select one stock | `simulate_paths`, array slicing |
| 4 | Inspect opening exposures and run the hedge policy | `initial_exposures`, `hedge` |
| 5 | Understand opening cash and subsequent events | `result.ledger`, `.loc`, `.iloc` |
| 6–7 | Reconcile cash, marked value and final settlement | NumPy assertions and cash-flow tables |
| 8 | Isolate the effect of stock transaction costs | Matched zero-cost and costly runs |
| 9 | Summarize all simulated outcomes | `summarize`, histograms |
| 10 | Record inputs, policy choices and source versions | `provenance` |

The hedge logic lives in `hedging_harness/core.py`. Pricing and Greeks reuse the consolidated Black–Scholes module. The markdown below explains each calculation and the resulting account entries.

## 1 · Connect to the project

Locate the project root and put it on Python's import search path. This works with the kernel started in either `MonteCarloEvaluations` or `Hedging_Study_Notebooks`.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('01_single_option_ledger')


## 2 · Select a named option and hedge policy

The dictionaries below are loaded from the shared validated study file, preserving the explicit association between each name and its inputs. `notional / spot` gives the positive magnitude of the short-call quantity.

- `OPTION_NAME` selects the position to trade.
- `HEDGE_EVERY = 5` means rebalance every five sessions; `None` means hedge initially and then hold the stock until expiry.
- `COST_BPS = 5.0` means a one-way stock trading cost of `5 / 10000 = 0.0005`, or $5 per $10,000 traded.
- `AUDIT_PATH` selects the single simulated outcome whose full daily ledger is retained. All paths still contribute to the terminal distribution.

The hedge uses the pricing volatility for delta. Changing hedge-volatility assumptions is a later sensitivity exercise.

## 3 · Generate study paths, then select one stock

Generate the **same joint three-name paths as notebook 00**, in the same name order, and select the chosen asset along the last axis. Reusing the seed with a different number of assets would change the random draws, so a separate one-name simulation would not reproduce the same histories.

The named correlation inputs are converted to a labelled matrix, then to an array at the simulator boundary. `single_paths` retains the asset axis, giving shape `(5000, 64, 1)` for the baseline. `single` contains only the selected option's contract inputs.

## 4 · Inspect exposures and run the hedge policy

The short calls have negative delta, so the initial offset is a **positive stock holding**. `initial_exposures(single)` displays the premium, hedge shares and remaining gamma/vega using the same conventions as notebook 00.

`hedge(...)` applies the policy to every selected path. It receives the model premium at inception, finances stock purchases and costs through cash, and accrues positive or negative cash at the same constant rate. Later trades use only the current spot and remaining maturity.

At expiry, the account liquidates its shares and pays the call's cash-settled payoff. The remaining cash is terminal net P&L. No additional capital is injected after inception.

## 5 · Read the opening cash account and subsequent events

At inception, the cash balance is

$$\text{cash}_0 = \text{premium received} - \text{shares purchased}\times S_0 - \text{entry cost}.$$

A negative balance means borrowing. The stock hedge generally costs more than the option premium received. The option liability remains on the account, so receiving a premium does not create an immediate profit.

The opening marked net value is `cash + stock value - option liability`; with a model-priced opening trade, this equals minus the entry stock cost. The exposure helper and the ledger should agree on both premium and hedge shares.

### Follow one realized path

Each session first accrues interest on the previous cash balance. The account then observes spot and either holds, rebalances, or settles:

- **open:** receive the premium and buy the initial stock hedge.
- **hold:** retain the previous shares; cash still accrues interest.
- **rebalance:** change shares to the current delta target, paying stock costs on the absolute traded value.
- **settle:** sell any remaining shares, charge liquidation costs and pay the call payoff.

A positive `trade_units` means buying stock and therefore a negative `stock_cashflow`. A hedge chosen at session t is held across the next interval. Final settlement happens at expiry even when that session is not a scheduled rebalance.

The tables separate holdings, cash flows and marked values so each accounting step is readable. The complete audit history remains available as `ledger`.

## 6 · Reconcile the account

Every row must satisfy the cash identity:

$$\Delta\text{cash} = \text{premium} + \text{interest} + \text{stock cash flow} - \text{cost} - \text{payoff paid}.$$

Accumulating those flows from an opening cash balance of zero reconstructs the cash account. Marked net value must equal cash plus stock value minus option liability. Stock trades, transaction costs and interest are also checked against their definitions.

Before expiry, the option liability is marked at the fixed pricing volatility. It is a model mark, rather than a changing market-implied-volatility observation. At expiry, the option is settled and the remaining liability and shares must both be zero.

### Inspect spot, hedge holdings, cash and marked value

The shares plot uses steps with `where="post"`: the new holding applies after the displayed trade time. Cash can be negative throughout much of a path because stock is financed with borrowing. Marked account value also includes the option liability, so cash alone is not interim P&L.

## 7 · Explain final settlement

On the last session, accrue interest on the previous cash balance, liquidate the stock, charge the liquidation cost, and pay `units * max(S_T - K, 0)`. The remaining cash is the terminal net P&L, since no stock or option liability remains.

The following waterfall lists signed cash contributions. The call payoff is a negative cash flow for the short position. The cost has already been charged through cash; it must not be subtracted again from terminal P&L.

## 8 · Isolate the effect of transaction costs

Run the **same hedge policy on exactly the same paths**, changing only the stock trading cost to zero. Trades are identical because the delta rule does not depend on cash or transaction costs in this model.

For every outcome:

$$\text{zero-cost P\&L} - \text{costly P\&L} = \text{costs accumulated to expiry at the financing rate}.$$

Nominal costs sum fees when charged. Financed costs also include the terminal effect of paying those fees earlier. With a positive rate, financed costs are at least nominal costs. Their difference is already reflected in cash and net P&L.

## 9 · Inspect the distribution across all paths

Each outcome is an individual trading result. `summarize(...)` reports dollar metrics for the terminal P&L distribution:

- `mean_pnl`: average simulated net P&L; `mean_se`: Monte Carlo uncertainty in that average.
- `pnl_std`: variability of trading outcomes within this model.
- `rmse`: root mean squared P&L relative to zero.
- `pnl_p05`: the fifth percentile of P&L.
- `loss_es95`: minus the mean of the worst 5% of outcomes, rounded up to a whole observation count. It can be negative if even that tail is profitable.
- Cost, turnover and trade-count columns describe implementation activity. Turnover includes entry and liquidation.

Mean SE is not economic risk and is not an uncertainty interval for tail losses. The zero-cost and costly outcomes are paired because the same paths are used. This notebook illustrates one policy; frequency comparisons belong in notebook 02.

## 10 · Record the actual experiment

The record includes the **joint study used for path generation**, the selected single-name trading book, the seed, path count, audit path, hedge interval, cost assumption and hedge volatility. Source hashes include the consolidated Black–Scholes module and accounting harness.

The notebook code hash excludes execution counts and rendered outputs. The record is displayed here; no additional export is performed. The run record identifies the exact shared study-file hash and the explicitly chosen policy parameters.

## What this notebook establishes

The accounting reconciles from opening premium through borrowing/lending, stock rebalances, costs, liquidation and call settlement. The final account has no shares or option liability, so its cash equals terminal net P&L. The matched zero-cost comparison isolates the financed effect of stock trading costs without subtracting costs twice.

Initial delta neutrality does not eliminate gamma, discrete-hedging error or transaction costs. The displayed P&L distribution remains conditional on illustrative Q-model assumptions. Dividends, option spreads, margin and exercise/assignment are outside the current model.

Next: **02_three_name_portfolio.ipynb** applies the accounting across names and compares hedge frequencies. That notebook loads the same shared model assumptions and declares its own policy controls.